# 🚀 Модель X-15: Экспериментальный гиперзвуковой самолёт

## 📖 Введение

Добро пожаловать в интерактивный пример работы с моделью **X-15** - легендарного экспериментального гиперзвукового самолёта! В этом notebook мы изучим:

- 🛩️ **Модель X-15**: Линейная продольная модель гиперзвукового самолёта
- 🎮 **Среда Gymnasium**: Создание и управление средой симуляции
- 📊 **Анализ данных**: Исследование состояний, управляющих воздействий и наград
- 🎯 **Система управления**: Понимание принципов управления гиперзвуковым полётом

X-15 был революционным самолётом, который достигал скоростей свыше 6 Махов и высот более 100 км, внося огромный вклад в развитие космонавтики и аэродинамики высоких скоростей.

---

## ⚙️ Настройка

Начнём с настройки рабочей среды и импорта необходимых библиотек.

### 📁 Настройка рабочей директории

Переходим в корневую директорию проекта для корректного импорта модулей:

In [1]:
# 📁 Переход в корневую директорию проекта
print('🔄 Переходим в корневую директорию...')
%cd ..
print('✅ Рабочая директория настроена!')

🔄 Переходим в корневую директорию...
/Users/asmazaev/TensorAeroSpace-Origin/example
✅ Рабочая директория настроена!


/Users/asmazaev/TensorAeroSpace-Origin/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


### 📦 Импорт библиотек

Загружаем необходимые модули для работы с моделью X-15:

In [2]:
# 🎮 Основные библиотеки для работы со средой
import gymnasium as gym  # Фреймворк для сред обучения с подкреплением
import numpy as np  # Библиотека для численных вычислений
from tqdm import tqdm  # Прогресс-бары для циклов

# 🚀 Импорт TensorAeroSpace для регистрации окружений
import tensoraerospace

# 🚀 Специфичные модули TensorAeroSpace для X-15
from tensoraerospace.envs import LinearLongitudinalX15  # Модель X-15
from tensoraerospace.utils import generate_time_period, convert_tp_to_sec_tp  # Утилиты времени
from tensoraerospace.signals.standart import unit_step  # Генератор ступенчатых сигналов

# 📊 Информация о загруженных модулях
print('📦 Библиотеки успешно загружены!')
print('🚀 Модель X-15 готова к использованию!')
print('🎯 Среда для гиперзвукового полёта настроена!')

📦 Библиотеки успешно загружены!
🚀 Модель X-15 готова к использованию!
🎯 Среда для гиперзвукового полёта настроена!


### ⚙️ Параметры симуляции

Настраиваем временные параметры и опорные сигналы для симуляции гиперзвукового полёта X-15:

In [3]:
# ⏱️ Временные параметры симуляции
dt = 0.01  # Шаг дискретизации (секунды)
tp = generate_time_period(tn=20, dt=dt)  # Временной период (20 секунд)
tps = convert_tp_to_sec_tp(tp, dt=dt)  # Преобразование в секунды
number_time_steps = len(tp)  # Общее количество временных шагов

# 🎯 Создание опорного сигнала (ступенчатый сигнал)
reference_signals = np.reshape(
    unit_step(degree=5, tp=tp, time_step=10, output_rad=True), 
    [1, -1]
)

# 📊 Информация о параметрах симуляции
print('⏱️ Параметры времени:')
print(f'   📏 Шаг дискретизации: {dt} сек')
print(f'   ⏰ Общее время симуляции: {tp[-1]:.1f} сек')
print(f'   🔢 Количество шагов: {number_time_steps}')
print()
print('🎯 Опорный сигнал:')
print(f'   📐 Амплитуда ступеньки: {np.degrees(5):.1f}°')
print(f'   ⏰ Время активации: {10 * dt:.1f} сек')
print(f'   📊 Размерность сигнала: {reference_signals.shape}')

⏱️ Параметры времени:
   📏 Шаг дискретизации: 0.01 сек
   ⏰ Общее время симуляции: 2002.0 сек
   🔢 Количество шагов: 2002

🎯 Опорный сигнал:
   📐 Амплитуда ступеньки: 286.5°
   ⏰ Время активации: 0.1 сек
   📊 Размерность сигнала: (1, 2002)


### 🚀 Создание среды X-15

Инициализируем среду симуляции гиперзвукового самолёта X-15 с заданными параметрами:

In [4]:
# 🚀 Создание среды симуляции X-15
print('🔧 Создаём среду LinearLongitudinalX15-v0...')

env = gym.make(
    'LinearLongitudinalX15-v0',
    number_time_steps=number_time_steps,
    initial_state=[[0], [0], [0], [0]],  # Полное начальное состояние модели [u, w, q, θ]
    reference_signal=reference_signals,
    state_space=["theta", "q"],  # Пространство состояний для наблюдения (по умолчанию)
    output_space=["theta", "q"],  # Выходное пространство
    tracking_states=["theta"]  # Отслеживаемые состояния
)

# 🔄 Сброс среды в начальное состояние
initial_observation, info = env.reset()

# 📊 Анализ начального состояния X-15
print('✅ Среда X-15 успешно создана!')
print()
print('🛩️ Начальное наблюдение X-15:')
print(f'   📏 Размерность наблюдения: {initial_observation.shape}')
print(f'   🏃 Продольная скорость (u): {initial_observation[0, 0]:.6f} м/с')
print(f'   ⬇️ Нормальная скорость (w): {initial_observation[1, 0]:.6f} м/с')
print(f'   🔄 Угловая скорость тангажа (q): {initial_observation[2, 0]:.6f} рад/с')
print(f'   📐 Угол тангажа (θ): {initial_observation[3, 0]:.6f} рад')
print()
print('💡 Примечание: Полная модель использует состояние [u, w, q, θ]')
print('   Наблюдение содержит все 4 состояния модели')
print()
print('🎯 X-15 готов к гиперзвуковому полёту!')

🔧 Создаём среду LinearLongitudinalX15-v0...
✅ Среда X-15 успешно создана!

🛩️ Начальное наблюдение X-15:
   📏 Размерность наблюдения: (4, 1)
   🏃 Продольная скорость (u): 0.000000 м/с
   ⬇️ Нормальная скорость (w): 0.000000 м/с
   🔄 Угловая скорость тангажа (q): 0.000000 рад/с
   📐 Угол тангажа (θ): 0.000000 рад

💡 Примечание: Полная модель использует состояние [u, w, q, θ]
   Наблюдение содержит все 4 состояния модели

🎯 X-15 готов к гиперзвуковому полёту!


/Users/asmazaev/TensorAeroSpace-Origin/.venv/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:159: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


### 🎮 Выполнение шага симуляции

Применяем управляющее воздействие к X-15 и анализируем результат:

In [5]:
# 🎮 Применение управляющего воздействияcontrol_input = np.array([[1]], dtype=np.float32)  # Управляющий сигнал (отклонение руля высоты в градусах)print('🎯 Применяем управляющее воздействие к X-15...')print(f'🕹️ Управляющий сигнал: {control_input[0][0]:.3f} град')print('⚡ Выполняем шаг симуляции...')# 🚀 Выполнение одного шага симуляцииobservation, reward, terminated, truncated, info = env.step(control_input)# 📊 Анализ результатов шагаprint('✅ Шаг симуляции выполнен!')print()print('📈 Результаты симуляции:')print(f'   🎯 Управляющий сигнал: {control_input[0][0]:.3f} град')print(f'   🏁 Эпизод завершён (terminated): {terminated}')print(f'   ⏱️ Эпизод обрезан (truncated): {truncated}')# Обработка награды (может быть массивом или скаляром)if isinstance(reward, np.ndarray):    reward_value = reward[0] if reward.size > 0 else reward.item()else:    reward_value = rewardprint(f'   🎁 Награда: {reward_value:.2e}')print(f'   📊 Размерность наблюдения: {observation.shape}')print()print('🛩️ X-15 успешно отреагировал на управление!')

### 🔍 Анализ внутренних данных модели

Исследуем историю управляющих воздействий, сохранённую в модели X-15:

In [6]:
# 🔍 Получение истории управляющих воздействий
input_history = env.model.store_input

# 📊 Анализ истории входных сигналов
print('🔍 Анализ истории управляющих воздействий X-15:')
print(f'📏 Размерность истории: {input_history.shape}')
print(f'🔢 Количество ненулевых входов: {np.count_nonzero(input_history)}')
print(f'🎯 Первые 5 значений: {input_history[0, :5]}')
print(f'📈 Максимальное значение: {np.max(input_history):.3f}')
print(f'📉 Минимальное значение: {np.min(input_history):.3f}')
print()
print('💡 История показывает применённые управляющие сигналы')
print('   к рулю высоты X-15 на каждом временном шаге')

input_history

🔍 Анализ истории управляющих воздействий X-15:
📏 Размерность истории: (1, 2002)
🔢 Количество ненулевых входов: 0
🎯 Первые 5 значений: [0. 0. 0. 0. 0.]
📈 Максимальное значение: 0.000
📉 Минимальное значение: 0.000

💡 История показывает применённые управляющие сигналы
   к рулю высоты X-15 на каждом временном шаге


/Users/asmazaev/TensorAeroSpace-Origin/.venv/lib/python3.10/site-packages/gymnasium/core.py:311: UserWarning: WARN: env.model to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.model` for environment variables or `env.get_wrapper_attr('model')` that will search the reminding wrappers.
  logger.warn(


array([[0., 0., 0., ..., 0., 0., 0.]])

### 📊 Анализ состояния X-15

Исследуем текущее состояние гиперзвукового самолёта после применения управляющего воздействия:

In [7]:
# 📊 Анализ текущего состояния X-15# Используем observation из предыдущей ячейки, если доступнаif 'observation' in locals() or 'observation' in globals():    current_observation = observationelse:    # Если observation не определена, получаем из env    current_observation, _ = env.reset()    observation, reward, terminated, truncated, info = env.step(np.array([[1]], dtype=np.float32))    current_observation = observation# 🛩️ Детальный разбор состоянияprint('📊 Анализ состояния X-15 после управляющего воздействия:')print(f'📏 Размерность наблюдения: {current_observation.shape}')print()# Компоненты состояния: [u, w, q, θ]state_names = ['u (продольная скорость)', 'w (нормальная скорость)', 'q (угловая скорость тангажа)', 'θ (угол тангажа)']print('🔍 Компоненты наблюдения:')for i, (name, value) in enumerate(zip(state_names, current_observation)):    print(f'   📈 {name}: {value[0]:.2e}')print()print('💡 Наблюдение показывает изменения в динамике X-15')print('   после применения управляющего сигнала к рулю высоты')print('🚀 Значения в научной нотации указывают на малые изменения')print('   что характерно для начальных моментов управления')current_observation

### 🏆 Анализ системы наград

Исследуем награду, полученную X-15 за выполненное управляющее воздействие:

In [8]:
# 🏆 Анализ системы наград X-15# Используем reward из предыдущей ячейки, если доступнаif 'reward' in locals() or 'reward' in globals():    current_reward = rewardelse:    # Если reward не определена, получаем из env    _, current_reward, _, _, _ = env.step(np.array([[1]], dtype=np.float32))# Обработка награды (может быть массивом или скаляром)if isinstance(current_reward, np.ndarray):    current_reward_value = current_reward[0] if current_reward.size > 0 else current_reward.item()else:    current_reward_value = current_reward# 🔍 Интерпретация наградыprint('🏆 Анализ награды за управление X-15:')print(f'🎯 Значение награды: {current_reward_value:.2e}')    if abs(current_reward_value) > 0:    print(f'📊 Порядок величины: {np.log10(abs(current_reward_value)):.1f}')print()# 💡 Интерпретация результатаif current_reward_value > 0:    print('✅ Положительная награда - хорошее управление!')    print('🚀 X-15 движется в правильном направлении')    print('📈 Система управления работает эффективно')elif current_reward_value == 0:    print('⚖️ Нейтральная награда - стабильное состояние')    print('🛩️ X-15 поддерживает текущий режим полёта')else:    print('❌ Отрицательная награда - требуется коррекция')    print('🔧 Необходимо улучшить управляющие воздействия')print()print('💡 Интерпретация для X-15:')print(f'   Награда {current_reward_value:.2e} отражает качество')print('   отслеживания опорного сигнала гиперзвуковым самолётом')print('🎯 Малое значение характерно для начальных этапов управления')current_reward_value

## 🎯 Заключение и следующие шаги

### 📚 Что мы изучили:

1. **🚀 Модель X-15** - экспериментальный гиперзвуковой самолёт
2. **🏋️ Среда Gymnasium** - интерфейс для обучения с подкреплением
3. **📊 Анализ состояния** - вектор наблюдений [u, w, q, θ] (все состояния модели)
4. **🎮 Система управления** - одномерные управляющие воздействия (отклонение руля высоты в градусах)
5. **🏆 Система наград** - оценка качества управления на основе отслеживания опорного сигнала
6. **📈 Мониторинг** - отслеживание внутренних данных модели

### 🔍 Важные детали:

- **Полное состояние модели**: [u, w, q, θ] где:
  - u - продольная скорость (м/с)
  - w - нормальная скорость (м/с)
  - q - угловая скорость тангажа (рад/с)
  - θ - угол тангажа (рад)
- **Пространство наблюдений**: В текущей реализации возвращаются все 4 состояния [u, w, q, θ]
- **API Gymnasium**: `reset()` возвращает `(observation, info)`, `step()` возвращает `(observation, reward, terminated, truncated, info)`
- **Управляющие воздействия**: Отклонение руля высоты в градусах (диапазон: ±60 градусов)

### 🚀 Следующие шаги:

1. **🤖 Обучение RL-агента** - создание интеллектуального пилота
2. **⚙️ Настройка параметров** - оптимизация симуляции
3. **🛩️ Сложные траектории** - тестирование различных манёвров
4. **📊 Визуализация** - графики состояний и управления
5. **🔬 Анализ производительности** - метрики качества управления

### 📖 Полезные ресурсы:

- [Reinforcement Learning: An Introduction](http://incompleteideas.net/book/the-book-2nd.html) - классический учебник по RL
- [Gymnasium Documentation](https://gymnasium.farama.org/) - документация по средам RL
- [TensorAeroSpace](https://github.com/TensorAeroSpace/TensorAeroSpace) - библиотека аэрокосмических сред

---

🎉 **Поздравляем!** Вы успешно освоили основы работы с гиперзвуковым самолётом X-15 в TensorAeroSpace. 
Используйте этот notebook как отправную точку для ваших исследований в области управления летательными аппаратами!